# paintingReorganize — GPU notebook

Rearranges a painting's pixels into a smooth palette. **Same pixels, same
rectangle**, reorganised.

**First: Runtime → Change runtime type → T4 GPU.** Then Runtime → Run all.

## You tune exactly two things

| knob | default | what it is |
|---|---|---|
| `SWEEPS` | 6000 | quality/time budget |
| `LAM` | 12 | composition: how strongly the palette is laid out as a sweep |

Everything else is calibrated automatically or fixed by design (kernel
reach spans the whole image; starting temperature is measured from the
energy landscape; stage-1 window is 3%). An expert appendix at the bottom
explains the hidden parameters, but the intended workflow is: **run the
defaults, look at the result, then consult the symptom table below.**

## Symptom → action

| what you see | what to change |
|---|---|
| result looks *worse* than the fast seed, chaotic | `SWEEPS` too low — the anneal melts before it rebuilds; runs must pass ~2500 to recover. Raise it. |
| blobby, composition drifting toward a bullseye | raise `LAM` (try 25) |
| colour regions pinched apart, over-striped, fragmented | lower `LAM` (try 6) |
| fine speckle at 1:1 zoom in colour-transition zones | mostly inherent (palette gaps must land somewhere); more `SWEEPS` helps a little |
| smooth but boring, too close to a plain gradient | lower `LAM` and let the pairwise term do more |


In [ ]:
!nvidia-smi -L || echo "NO GPU - Runtime > Change runtime type > T4 GPU"
import torch; print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# setup
import os
if not os.path.isdir('repo'):
    !git clone -q --branch claude/brave-newton-xsqgpe https://github.com/ardila/paintingReorganize.git repo
%cd repo
!git pull -q
!pip -q install scipy pillow imageio imageio-ffmpeg
import importlib, numpy as np, time
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import smooth_palette_gpu as G
importlib.reload(G)
import matplotlib.pyplot as plt
print("ready")

## Run

Set `SRC` to any painting in the repo (`demoiselles.jpg`,
`starry_night.png`, `the_large_bathers.jpg`, `input.jpg`) or upload your
own via the Files panel. `SCALE=0.5` while iterating; `1.0` for finals.

In [ ]:
SRC    = 'demoiselles.jpg'
SCALE  = 0.5
SWEEPS = 6000
LAM    = 12.0

im = Image.open(SRC).convert('RGB')
if SCALE != 1.0:
    im = im.resize((int(im.width*SCALE), int(im.height*SCALE)), Image.LANCZOS)
rgb = np.asarray(im)

t = time.time()
out = G.run(rgb, sweeps=SWEEPS, lam=LAM)
print(f"{(time.time()-t)/60:.1f} min")

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(rgb); ax[0].set_title('original')
ax[1].imshow(out); ax[1].set_title(f'result (SWEEPS={SWEEPS}, LAM={LAM})')
y, x = out.shape[0]//2, out.shape[1]//2
ax[2].imshow(out[max(0,y-150):y+150, max(0,x-150):x+150]); ax[2].set_title('1:1 crop - check for grain here')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()
Image.fromarray(out).save('result.png')

## Compare a handful of LAM values (optional)

If the symptom table says adjust `LAM`, this shows the whole dial at
once instead of guessing one value at a time.

In [ ]:
candidates = (6.0, 12.0, 25.0)
outs = []
for l in candidates:
    t = time.time(); o = G.run(rgb, sweeps=min(SWEEPS, 3000), lam=l, verbose=False)
    outs.append((l, o)); print(f"lam={l}: {time.time()-t:.0f}s")
fig, ax = plt.subplots(1, len(outs), figsize=(6*len(outs), 6))
for i,(l,o) in enumerate(outs):
    ax[i].imshow(o); ax[i].set_title(f'LAM={l}'); ax[i].axis('off')
plt.tight_layout(); plt.show()

## Video of the evolution

The anneal melts the seed and rebuilds it while cooling — it *should*
look terrible in the middle.

In [ ]:
import imageio.v2 as imageio
writer = imageio.get_writer('evolution.mp4', fps=30, quality=8, macro_block_size=1)
out_v = G.run(rgb, sweeps=SWEEPS, lam=LAM, verbose=False,
              frame_every=max(1, SWEEPS//600),
              on_frame=lambda i, f: writer.append_data(f))
for _ in range(60): writer.append_data(out_v)
writer.close()
from IPython.display import Video, display
display(Video('evolution.mp4', embed=True, width=640))
from google.colab import files; files.download('evolution.mp4')

## Garden with Peacocks, full 10.6 MP

In [ ]:
!wget -q -O peacocks.jpg "https://commons.wikimedia.org/wiki/Special:FilePath/Franti%C5%A1ek_Kupka_%E2%80%93_Garden_with_Peacocks.jpg?width=3609"
rgb_p = np.asarray(Image.open('peacocks.jpg').convert('RGB'))
t = time.time()
out_p = G.run(rgb_p, sweeps=6000, lam=LAM)
print(f"{(time.time()-t)/60:.1f} min")
Image.fromarray(out_p).save('peacocks_smooth.png')
plt.figure(figsize=(16,13)); plt.imshow(out_p); plt.axis('off'); plt.show()
from google.colab import files; files.download('peacocks_smooth.png')

## Appendix: the hidden parameters, for the curious

You should not need these. They exist, they have reasons, and their
defaults were either measured or fixed by design:

- **kernel reach** — the pairwise attraction follows ~1/r² out to the
  whole image (octave ladder of Gaussians, pyramid-accelerated). There
  used to be a `top_frac` knob capping the reach; it was removed because
  once the tail spans the picture there is nothing left to choose.
- **starting temperature** — measured per image: swap proposals are
  sampled at the seed and T0 is set so the median uphill move is
  accepted with probability `accept=0.25`. Pass `accept=` to `G.run` to
  melt more (0.6) or less (0.05); untested territory either way.
- **stage-1 window (3%)** — the noise-vs-2D dial of the initialisation.
  Matters much less since stage 2 reorganises structure anyway.
- **hollow core** — the kernel is repulsive under ~2.5px so residual
  colour variance stays as invisible 1px dither instead of visible
  clumps. Structural, not a matter of taste.
